In [2]:
""" 
Convolution

    Implemented in conv_forward and conv_backward.

    Supports arbitrary stride & padding (here we use stride=1, pad=1 for “same” conv).

Pooling (Max)

    Implemented in maxpool_forward (2×2, stride=2) and maxpool_backward.

Fully Connected Layers

    In the final stage, after flattening, we have two FC layers:

        FC1 (hidden dim = 64) + ReLU + (optional BatchNorm + Dropout)

        FC2 (output dim = 3) → raw class logits

Padding & Stride

    The conv code honors the pad and stride arguments to determine output spatial size.

    In this example, we fixed them to pad=1, stride=1 for each conv to preserve “same” spatial dims.

Dropout

    After each hidden FC‐ReLU, we apply dropout with dropout_prob=0.3 during training.

    The mask is stored in caches["dropout_mask{i}"] and used in backprop.

Flatten

    After the last pooling layer, we reshape (N, C, H, W) → (N, C*H*W) in forward.

BatchNorm

    We apply BatchNorm immediately after each conv (before ReLU) and after each hidden FC (before ReLU).

    Both “conv‐BN” and “FC‐BN” are handled by BatchNormLayer.

Transfer Learning

    Not explicitly coded, but you could easily replace the first few conv layers by loading pre‐trained weights (e.g. from a larger model) and freeze them.

    See notes below on how to extend.

Feature Visualization

    Also not coded here—but you can take any learned filter in W_conv{i} or any activation map in A_conv{i} and visualize it (e.g. plot W_conv1[0,0] as an 8×8 heatmap).

    See notes below.

Triplet Loss (Metric Learning)

    Not used in the softmax‐cross‐entropy loop. To add triplet loss, you’d build a parallel “embedding” CNN that maps each image to a low‐dim vector, then compute triplet‐loss on (anchor, positive, negative) triples.

    See notes below.
"""

' \nConvolution\n\n    Implemented in conv_forward and conv_backward.\n\n    Supports arbitrary stride & padding (here we use stride=1, pad=1 for “same” conv).\n\nPooling (Max)\n\n    Implemented in maxpool_forward (2×2, stride=2) and maxpool_backward.\n\nFully Connected Layers\n\n    In the final stage, after flattening, we have two FC layers:\n\n        FC1 (hidden dim = 64) + ReLU + (optional BatchNorm + Dropout)\n\n        FC2 (output dim = 3) → raw class logits\n\nPadding & Stride\n\n    The conv code honors the pad and stride arguments to determine output spatial size.\n\n    In this example, we fixed them to pad=1, stride=1 for each conv to preserve “same” spatial dims.\n\nDropout\n\n    After each hidden FC‐ReLU, we apply dropout with dropout_prob=0.3 during training.\n\n    The mask is stored in caches["dropout_mask{i}"] and used in backprop.\n\nFlatten\n\n    After the last pooling layer, we reshape (N, C, H, W) → (N, C*H*W) in forward.\n\nBatchNorm\n\n    We apply BatchNorm 

In [5]:
import numpy as np

# ----------------------------------------
# 1) Synthetic “Image” Dataset (unchanged)
# ----------------------------------------
def generate_synthetic_images(num_classes=3,
                              samples_per_class=200,
                              img_size=8,
                              seed=0):
    np.random.seed(seed)
    N = num_classes * samples_per_class
    H = W = img_size
    X = np.zeros((N, 1, H, W))
    Y = np.zeros((N, num_classes))
    for c in range(num_classes):
        for i in range(samples_per_class):
            idx = c * samples_per_class + i
            img = np.random.randn(H, W) * 0.1
            if c == 0:
                img[:H//2, :W//2] += 1.0
            elif c == 1:
                img[:H//2, W//2:] += 1.0
            else:
                img[H//2:, :W//2] += 1.0
            X[idx, 0] = img
            Y[idx, c] = 1.0
    perm = np.random.permutation(N)
    return X[perm], Y[perm]

# ----------------------------------------
# 2) BatchNorm (unchanged)
# ----------------------------------------
class BatchNormLayer:
    def __init__(self, num_features, eps=1e-5, momentum=0.9, is_conv=False):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        self.is_conv = is_conv
        
        # Learnable scale & shift
        self.gamma = np.ones((num_features,))
        self.beta  = np.zeros((num_features,))
        
        # Running (EMA) stats
        self.running_mean = np.zeros((num_features,))
        self.running_var  = np.ones((num_features,))
        
        # Cache for backward
        self.cache = None

    def forward(self, X, training=True):
        if self.is_conv:
            N, C, H, W = X.shape
            if training:
                mu = X.mean(axis=(0,2,3))            # shape (C,)
                var = X.var(axis=(0,2,3))            # shape (C,)
                self.running_mean = (self.momentum * self.running_mean +
                                     (1 - self.momentum) * mu)
                self.running_var  = (self.momentum * self.running_var +
                                     (1 - self.momentum) * var)
            else:
                mu, var = self.running_mean, self.running_var
            
            # Reshape mu/var for broadcast
            mu_resh = mu.reshape((1, C, 1, 1))        # (1, C, 1, 1)
            var_resh = var.reshape((1, C, 1, 1))      # (1, C, 1, 1)
            
            X_centered = X - mu_resh                  # (N, C, H, W)
            inv_std = 1.0 / np.sqrt(var + self.eps)   # (C,)
            inv_std_resh = inv_std.reshape((1, C, 1, 1))  # (1, C, 1, 1)
            X_hat = X_centered * inv_std_resh         # (N, C, H, W)
            out = (self.gamma.reshape((1, C, 1, 1)) * X_hat +
                   self.beta.reshape((1, C, 1, 1)))    # (N, C, H, W)
            
            if training:
                # Cache exactly what we need in backward
                # Store inv_std as 1D (C,) and also keep X_centered, var, and M
                M = N * H * W
                self.cache = (X_hat, inv_std, X_centered, var, M)
            return out

        else:
            N, D = X.shape
            if training:
                mu = X.mean(axis=0)       # (D,)
                var = X.var(axis=0)       # (D,)
                self.running_mean = (self.momentum * self.running_mean +
                                     (1 - self.momentum) * mu)
                self.running_var  = (self.momentum * self.running_var +
                                     (1 - self.momentum) * var)
            else:
                mu, var = self.running_mean, self.running_var
            
            mu_resh = mu.reshape((1, D))         # (1, D)
            var_resh = var.reshape((1, D))       # (1, D)
            X_centered = X - mu_resh             # (N, D)
            inv_std = 1.0 / np.sqrt(var + self.eps)   # (D,)
            inv_std_resh = inv_std.reshape((1, D))    # (1, D)
            X_hat = X_centered * inv_std_resh     # (N, D)
            out = (self.gamma.reshape((1, D)) * X_hat +
                   self.beta.reshape((1, D)))    # (N, D)
            
            if training:
                M = N
                self.cache = (X_hat, inv_std, X_centered, var, M)
            return out

    def backward(self, dout):
        if self.is_conv:
            # Unpack cached values
            X_hat, inv_std, X_centered, var, M = self.cache
            # dout shape: (N, C, H, W)
            N, C, H, W = dout.shape
            
            # Reshape inv_std, etc. for proper broadcast
            inv_std_resh = inv_std.reshape((1, C, 1, 1))      # (1, C, 1, 1)
            X_centered_resh = X_centered                      # (N, C, H, W)
            
            # dgamma, dbeta (shape (C,))
            dgamma = np.sum(dout * X_hat, axis=(0, 2, 3))
            dbeta  = np.sum(dout, axis=(0, 2, 3))
            
            # Backprop through normalization:
            # Step 1: dX_hat = dout * gamma
            dX_hat = dout * self.gamma.reshape((1, C, 1, 1))  # (N, C, H, W)
            
            # Step 2: dvar = sum(dX_hat * X_centered * (-0.5) * inv_std^3)
            # inv_std^3: shape (C,) → reshape to (1, C, 1, 1)
            inv_std3 = (inv_std**3).reshape((1, C, 1, 1))
            dvar = np.sum(dX_hat * X_centered_resh * (-0.5) * inv_std3,
                          axis=(0, 2, 3))  # (C,)
            
            # Step 3: dmu = sum(dX_hat * -inv_std) + dvar * mean(-2 * X_centered)
            dmu_part1 = np.sum(dX_hat * (-inv_std_resh), axis=(0, 2, 3))  # (C,)
            dmu_part2 = dvar * np.mean(-2 * X_centered_resh, axis=(0, 2, 3))  # (C,)
            dmu = dmu_part1 + dmu_part2  # (C,)
            
            # Step 4: dX
            term1 = dX_hat * inv_std_resh                          # (N, C, H, W)
            term2 = (dvar.reshape((1, C, 1, 1)) * 2 * X_centered_resh) / M  # (N, C, H, W)
            term3 = dmu.reshape((1, C, 1, 1)) / M                   # (1, C, 1, 1) broadcast over (N, C, H, W)
            dX = term1 + term2 + term3                             # (N, C, H, W)
            
            self.dgamma = dgamma
            self.dbeta  = dbeta
            return dX

        else:
            # Fully‐connected BN backward (2D)
            X_hat, inv_std, X_centered, var, M = self.cache
            N, D = X_hat.shape
            
            # dgamma, dbeta (shape (D,))
            dgamma = np.sum(dout * X_hat, axis=0)
            dbeta  = np.sum(dout, axis=0)
            
            # dX_hat = dout * gamma
            dX_hat = dout * self.gamma.reshape((1, D))   # (N, D)
            
            # dvar = sum(dX_hat * X_centered * (-0.5) * inv_std^3)
            inv_std3 = (inv_std**3).reshape((1, D))      # (1, D)
            dvar = np.sum(dX_hat * X_centered * (-0.5) * inv_std3, axis=0)  # (D,)
            
            # dmu = sum(dX_hat * -inv_std) + dvar * mean(-2 * X_centered)
            dmu_part1 = np.sum(dX_hat * (-inv_std.reshape((1, D))), axis=0)  # (D,)
            dmu_part2 = dvar * np.mean(-2 * X_centered, axis=0)               # (D,)
            dmu = dmu_part1 + dmu_part2  # (D,)
            
            # dX = dX_hat * inv_std + (dvar * 2*X_centered / M) + (dmu / M)
            term1 = dX_hat * inv_std.reshape((1, D))                    # (N, D)
            term2 = (dvar.reshape((1, D)) * 2 * X_centered) / M         # (N, D)
            term3 = dmu.reshape((1, D)) / M                              # (1, D), broadcast to (N, D)
            dX = term1 + term2 + term3                                   # (N, D)
            
            self.dgamma = dgamma
            self.dbeta  = dbeta
            return dX


# ----------------------------------------
# 3) Convolution + Pooling (unchanged)
# ----------------------------------------
def conv_forward(X, W, b, stride=1, pad=0):
    N, C_in, H_in, W_in = X.shape
    F, _, KH, KW = W.shape
    H_out = 1 + (H_in + 2*pad - KH) // stride
    W_out = 1 + (W_in + 2*pad - KW) // stride
    X_pad = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), 'constant')
    out = np.zeros((N, F, H_out, W_out))
    for n in range(N):
        for f in range(F):
            for i in range(H_out):
                for j in range(W_out):
                    vs, ve = i*stride, i*stride + KH
                    hs, he = j*stride, j*stride + KW
                    window = X_pad[n, :, vs:ve, hs:he]
                    out[n, f, i, j] = np.sum(window * W[f]) + b[f]
    cache = (X, W, b, stride, pad, X_pad)
    return out, cache

def conv_backward(dout, cache):
    X, W, b, stride, pad, X_pad = cache
    N, C_in, H_in, W_in = X.shape
    F, _, KH, KW = W.shape
    _, _, H_out, W_out = dout.shape
    dX_pad = np.zeros_like(X_pad)
    dW = np.zeros_like(W)
    db = np.zeros_like(b)
    for n in range(N):
        for f in range(F):
            for i in range(H_out):
                for j in range(W_out):
                    vs, ve = i*stride, i*stride + KH
                    hs, he = j*stride, j*stride + KW
                    window = X_pad[n, :, vs:ve, hs:he]
                    dW[f] += window * dout[n,f,i,j]
                    db[f] += dout[n,f,i,j]
                    dX_pad[n, :, vs:ve, hs:he] += W[f] * dout[n,f,i,j]
    dX = dX_pad[:, :, pad:pad+H_in, pad:pad+W_in]
    return dX, dW, db

def maxpool_forward(X, pool_height=2, pool_width=2, stride=2):
    N, C, H, W = X.shape
    H_out = 1 + (H - pool_height)//stride
    W_out = 1 + (W - pool_width)//stride
    out = np.zeros((N, C, H_out, W_out))
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    vs, ve = i*stride, i*stride + pool_height
                    hs, he = j*stride, j*stride + pool_width
                    window = X[n, c, vs:ve, hs:he]
                    out[n, c, i, j] = np.max(window)
    cache = (X, pool_height, pool_width, stride)
    return out, cache

def maxpool_backward(dout, cache):
    X, pool_height, pool_width, stride = cache
    N, C, H, W = X.shape
    _, _, H_out, W_out = dout.shape
    dX = np.zeros_like(X)
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    vs, ve = i*stride, i*stride + pool_height
                    hs, he = j*stride, j*stride + pool_width
                    window = X[n, c, vs:ve, hs:he]
                    mask = (window == np.max(window))
                    dX[n, c, vs:ve, hs:he] += mask * dout[n, c, i, j]
    return dX

# ----------------------------------------
# 4) SimpleCNN with the fix
# ----------------------------------------
class SimpleCNN:
    def __init__(self, input_dim=(1,8,8),
                 conv_params=[(4,1,3,1,1),(8,4,3,1,1)],
                 pool_params=[(2,2,2),(2,2,2)],
                 fc_dims=[64,3],
                 use_batchnorm=True,
                 dropout_prob=0.3,
                 lr=1e-2,
                 seed=0):
        np.random.seed(seed)
        self.lr = lr
        self.use_batchnorm = use_batchnorm
        self.dropout_prob = dropout_prob

        # ---- Build conv layers ----
        self.params = {}
        self.bn_conv = {}
        in_C, in_H, in_W = input_dim
        cur_C, cur_H, cur_W = in_C, in_H, in_W

        for idx, (F, C_in, K, stride, pad) in enumerate(conv_params, start=1):
            self.params[f"W_conv{idx}"] = np.random.randn(F, C_in, K, K) * np.sqrt(2.0/(C_in*K*K))
            self.params[f"b_conv{idx}"] = np.zeros((F,))
            if use_batchnorm:
                self.bn_conv[f"bn_conv{idx}"] = BatchNormLayer(num_features=F, is_conv=True)
            out_H = 1 + (cur_H + 2*pad - K)//stride
            out_W = 1 + (cur_W + 2*pad - K)//stride
            # apply pooling
            p_h, p_w, p_stride = pool_params[idx-1]
            out_H = 1 + (out_H - p_h)//p_stride
            out_W = 1 + (out_W - p_w)//p_stride
            cur_C, cur_H, cur_W = F, out_H, out_W

        # ---- Build FC layers ----
        self.bn_fc = {}
        prev_dim = cur_C * cur_H * cur_W
        for i, out_dim in enumerate(fc_dims, start=1):
            if i < len(fc_dims):
                self.params[f"W_fc{i}"] = np.random.randn(prev_dim, out_dim) * np.sqrt(2.0/prev_dim)
            else:
                self.params[f"W_fc{i}"] = np.random.randn(prev_dim, out_dim) * 0.01
            self.params[f"b_fc{i}"] = np.zeros((1, out_dim))
            if use_batchnorm and i < len(fc_dims):
                self.bn_fc[f"bn_fc{i}"] = BatchNormLayer(num_features=out_dim, is_conv=False)
            prev_dim = out_dim

    def forward(self, X, training=True):
        caches = {}
        A = X

        # -- Conv blocks --
        num_conv = len([k for k in self.params if k.startswith("W_conv")])
        for idx in range(1, num_conv+1):
            W = self.params[f"W_conv{idx}"]
            b = self.params[f"b_conv{idx}"]
            conv_out, conv_cache = conv_forward(A, W, b, stride=1, pad=1)
            caches[f"conv_cache{idx}"] = conv_cache

            if self.use_batchnorm:
                bn_layer = self.bn_conv[f"bn_conv{idx}"]
                bn_out = bn_layer.forward(conv_out, training=training)
                caches[f"bn_conv_cache{idx}"] = bn_layer.cache
                relu_in = bn_out
            else:
                relu_in = conv_out

            relu_out = np.maximum(0, relu_in)
            caches[f"relu_conv_cache{idx}"] = relu_in

            p_h, p_w, p_stride = (2,2,2)
            pool_out, pool_cache = maxpool_forward(relu_out, p_h, p_w, p_stride)
            caches[f"pool_cache{idx}"] = pool_cache
            A = pool_out

        # -- Flatten --
        N, C, H, W = A.shape
        A_flat = A.reshape(N, -1)
        caches["A_fc0"] = A_flat  # <--- store A_fc0
        caches["flat_shape"] = (C, H, W)

        # -- FC blocks --
        out = A_flat
        num_fc = len([k for k in self.params if k.startswith("W_fc")])
        for i in range(1, num_fc+1):
            W = self.params[f"W_fc{i}"]
            b = self.params[f"b_fc{i}"]
            Z = out.dot(W) + b
            caches[f"Z_fc{i}"] = Z
            if i < num_fc:
                if self.use_batchnorm:
                    bn_layer = self.bn_fc[f"bn_fc{i}"]
                    Z_norm = bn_layer.forward(Z, training=training)
                    caches[f"bn_fc_cache{i}"] = bn_layer.cache
                    act_in = Z_norm
                else:
                    act_in = Z
                A_i = np.maximum(0, act_in)
                caches[f"A_fc{i}"] = A_i  # <--- store A_fc{i}

                if training and self.dropout_prob > 0:
                    mask = (np.random.rand(*A_i.shape) > self.dropout_prob) / (1-self.dropout_prob)
                    A_i *= mask
                    caches[f"dropout_mask{i}"] = mask
                out = A_i
            else:
                out = Z
        return out, caches

    def compute_loss_and_grads(self, scores, Y, caches):
        grads = {}
        N, C = scores.shape
        shifted = scores - np.max(scores, axis=1, keepdims=True)
        exp_s = np.exp(shifted)
        probs = exp_s / np.sum(exp_s, axis=1, keepdims=True)
        loss = -np.sum(Y * np.log(probs + 1e-8)) / N

        dScores = (probs - Y) / N
        num_fc = len([k for k in self.params if k.startswith("W_fc")])
        dout = dScores
        for i in reversed(range(1, num_fc+1)):
            Z_i = caches[f"Z_fc{i}"]
            W = self.params[f"W_fc{i}"]
            A_prev = caches[f"A_fc{i-1}"]  # now exists!
            grads[f"dW_fc{i}"] = A_prev.T.dot(dout)
            grads[f"db_fc{i}"] = np.sum(dout, axis=0, keepdims=True)
            dZ = dout.dot(W.T)
            if i > 1:
                if f"dropout_mask{i-1}" in caches:
                    dZ *= caches[f"dropout_mask{i-1}"]
                act_in = caches[f"A_fc{i-1}"]
                dZ = dZ * (act_in > 0)
                if self.use_batchnorm:
                    bn_layer = self.bn_fc[f"bn_fc{i-1}"]
                    dZ = bn_layer.backward(dZ)
                    grads[f"dgamma_fc{i-1}"] = bn_layer.dgamma
                    grads[f"dbeta_fc{i-1}"]  = bn_layer.dbeta
                dout = dZ
            else:
                dout = dZ.reshape(N, *caches["flat_shape"])

        dA_pool = dout
        num_conv = len([k for k in self.params if k.startswith("W_conv")])
        for idx in reversed(range(1, num_conv+1)):
            pool_cache = caches[f"pool_cache{idx}"]
            dZ_relu = maxpool_backward(dA_pool, pool_cache)
            relu_in = caches[f"relu_conv_cache{idx}"]
            dZ_norm = dZ_relu * (relu_in > 0)
            if self.use_batchnorm:
                bn_layer = self.bn_conv[f"bn_conv{idx}"]
                dZ = bn_layer.backward(dZ_norm)
                grads[f"dgamma_conv{idx}"] = bn_layer.dgamma
                grads[f"dbeta_conv{idx}"]  = bn_layer.dbeta
            else:
                dZ = dZ_norm
            conv_cache = caches[f"conv_cache{idx}"]
            dA_prev, dW, db = conv_backward(dZ, conv_cache)
            grads[f"dW_conv{idx}"] = dW
            grads[f"db_conv{idx}"] = db
            dA_pool = dA_prev
        return loss, grads

    def update_params(self, grads):
        """
        Plain SGD update (no momentum).
        Only update keys that exist in self.params; each param 'X' expects a grad 'dX'.
        """
        for param_name in list(self.params.keys()):
            grad_name = "d" + param_name
            if grad_name in grads:
                self.params[param_name] -= self.lr * grads[grad_name]


    def predict(self, X):
        scores, _ = self.forward(X, training=False)
        return np.argmax(scores, axis=1)

# ----------------------------------------
# 5) Training Loop
# ----------------------------------------
if __name__ == "__main__":
    X, Y_onehot = generate_synthetic_images(num_classes=3,
                                            samples_per_class=200,
                                            img_size=8,
                                            seed=1)
    N = X.shape[0]
    split = int(0.8 * N)
    X_train, Y_train = X[:split], Y_onehot[:split]
    X_val,   Y_val   = X[split:], Y_onehot[split:]
    Y_val_int = np.argmax(Y_val, axis=1)

    conv_params = [(4, 1, 3, 1, 1), (8, 4, 3, 1, 1)]
    pool_params = [(2,2,2), (2,2,2)]
    fc_dims = [64, 3]

    model = SimpleCNN(input_dim=(1,8,8),
                      conv_params=conv_params,
                      pool_params=pool_params,
                      fc_dims=fc_dims,
                      use_batchnorm=True,
                      dropout_prob=0.3,
                      lr=1e-2,
                      seed=2)

    epochs = 20
    for epoch in range(1, epochs+1):
        scores, caches = model.forward(X_train, training=True)
        loss, grads = model.compute_loss_and_grads(scores, Y_train, caches)
        model.update_params(grads)
        if epoch % 5 == 0 or epoch == 1:
            train_preds = model.predict(X_train)
            train_acc = np.mean(train_preds == np.argmax(Y_train, axis=1))
            val_preds = model.predict(X_val)
            val_acc = np.mean(val_preds == Y_val_int)
            print(f"Epoch {epoch:2d} | Loss: {loss:.4f} | "
                  f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    train_acc = np.mean(model.predict(X_train) == np.argmax(Y_train, axis=1))
    val_acc = np.mean(model.predict(X_val) == Y_val_int)
    print(f"\nFinal Train Acc: {train_acc:.4f} | Final Val Acc: {val_acc:.4f}")


Epoch  1 | Loss: 1.1091 | Train Acc: 0.4042 | Val Acc: 0.4333
Epoch  5 | Loss: 0.9541 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 10 | Loss: 0.8006 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 15 | Loss: 0.6826 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 20 | Loss: 0.5792 | Train Acc: 1.0000 | Val Acc: 1.0000

Final Train Acc: 1.0000 | Final Val Acc: 1.0000
